In [8]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Literal,Annotated
from langchain_groq import ChatGroq
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from dotenv import load_dotenv

In [9]:
load_dotenv()

True

In [10]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [11]:
llm=ChatGroq(model="llama-3.1-8b-instant")
def chat_node(state:ChatState):
    messages=state['messages']
    response=llm.invoke(messages).content
    return {'messages':[response]}

In [12]:
checkpointer=MemorySaver()
graph=StateGraph(ChatState)
graph.add_node("chat_node",chat_node)
graph.add_edge(START,"chat_node")
graph.add_edge("chat_node",END)
chatbot=graph.compile(checkpointer=checkpointer)


In [14]:
thread_id='1'
while True:
    user_messages=input("User:")
    print('User:',user_messages)
    if user_messages.strip().lower() in ['exit','quit','bye']:
        break
    config={'configurable':{'thread_id':thread_id}}
    response=chatbot.invoke({'messages': [HumanMessage(content=user_messages)]},config=config)
    print('Bot:',response['messages'][-1].content)


User: hi
Bot: How can I assist you today?
User: my name is nikhil
Bot: Hello Nikhil, it's nice to meet you. I'm happy to assist you with any questions or topics you'd like to discuss. What's on your mind today?
User: whats my name?
Bot: Your name is Nikhil. We just had that confirmed a little while ago.
User: exit
